## Project setup

In [69]:
from pathlib import Path

import pandas as pd

In [70]:
PROJECT_DIR = Path.cwd().parent

RAW_DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'real_estate_raw.csv'
PROCESSED_DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'real_estate_clean.csv'

REPORTS_DIR = PROJECT_DIR / 'reports'
CLEANING_LOG_PATH = REPORTS_DIR / 'cleaning_log.csv'

## Data loading

In [71]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
df_raw = df.copy()

In [72]:
df.shape

(5477006, 13)

In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5477006 entries, 0 to 5477005
Data columns (total 13 columns):
 #   Column         Dtype  
---  ------         -----  
 0   price          int64  
 1   date           str    
 2   time           str    
 3   geo_lat        float64
 4   geo_lon        float64
 5   region         int64  
 6   building_type  int64  
 7   floor          int64  
 8   floors         int64  
 9   rooms          int64  
 10  area           float64
 11  kitchen_area   float64
 12  object_type    int64  
dtypes: float64(4), int64(7), str(2)
memory usage: 543.2 MB


In [74]:
df.describe()

,price,geo_lat,geo_lon,region,building_type,floor,floors,rooms,area,kitchen_area,object_type
count,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06,5.477006e+06
mean,4.422029e+06,5.403826e+01,5.324433e+01,4.307141e+03,1.948966e+00,6.214530e+00,1.139892e+01,1.726173e+00,5.391825e+01,1.062840e+01,3.945399e+00
std,2.150752e+07,4.622758e+00,2.074763e+01,3.308050e+03,1.038537e+00,4.957419e+00,6.535734e+00,1.082133e+00,3.335293e+01,9.792380e+00,4.558357e+00
min,-2.144967e+09,4.145906e+01,1.989020e+01,3.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,-2.000000e+00,7.000000e-02,1.000000e-02,1.000000e+00
25%,1.950000e+06,5.337768e+01,3.777790e+01,2.661000e+03,1.000000e+00,2.000000e+00,5.000000e+00,1.000000e+00,3.800000e+01,7.000000e+00,1.000000e+00
50%,2.990000e+06,5.517139e+01,4.306774e+01,2.922000e+03,2.000000e+00,5.000000e+00,1.000000e+01,2.000000e+00,4.802000e+01,9.700000e+00,1.000000e+00
75%,4.802000e+06,5.622613e+01,6.564895e+01,6.171000e+03,3.000000e+00,9.000000e+00,1.600000e+01,2.000000e+00,6.313000e+01,1.270000e+01,1.100000e+01
max,2.147484e+09,7.198040e+01,1.625361e+02,6.188800e+04,5.000000e+00,3.900000e+01,3.900000e+01,1.000000e+01,7.856000e+03,9.999000e+03,1.100000e+01


## Data types conversion

In [75]:
df['published_at'] = pd.to_datetime(
    df['date'] + ' ' + df['time'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

df['publication_year'] = df['published_at'].dt.year
df['publication_quarter'] = df['published_at'].dt.quarter
df['publication_month'] = df['published_at'].dt.month

In [76]:
df.head()

,price,date,time,geo_lat,geo_lon,region,building_type,floor,floors,rooms,area,kitchen_area,object_type,published_at,publication_year,publication_quarter,publication_month
0,6050000,2018-02-19,20:00:21,59.805808,30.376141,2661,1,8,10,3,82.6,10.8,1,2018-02-19 20:00:21,2018,1,2
1,8650000,2018-02-27,12:04:54,55.683807,37.297405,81,3,5,24,2,69.1,12.0,1,2018-02-27 12:04:54,2018,1,2
2,4000000,2018-02-28,15:44:00,56.295250,44.061637,2871,1,5,9,3,66.0,10.0,1,2018-02-28 15:44:00,2018,1,2
3,1850000,2018-03-01,11:24:52,44.996132,39.074783,2843,4,12,16,2,38.0,5.0,11,2018-03-01 11:24:52,2018,1,3
4,5450000,2018-03-01,17:42:43,55.918767,37.984642,81,3,13,14,2,60.0,10.0,1,2018-03-01 17:42:43,2018,1,3


## Missing values check

In [77]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values

price                  0
date                   0
time                   0
geo_lat                0
geo_lon                0
region                 0
building_type          0
floor                  0
floors                 0
rooms                  0
area                   0
kitchen_area           0
object_type            0
published_at           0
publication_year       0
publication_quarter    0
publication_month      0
dtype: int64

In [78]:
df = df.dropna(subset=["price", "area", "rooms"])

## Duplicates check 

In [79]:
duplicates_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicates_count)
print("Duplicate percent:", round(duplicates_count / df.shape[0] * 100, 2))

Exact duplicate rows: 1523
Duplicate percent: 0.03


In [80]:
df[df.duplicated()].head(10)

,price,date,time,geo_lat,geo_lon,region,building_type,floor,floors,rooms,area,kitchen_area,object_type,published_at,publication_year,publication_quarter,publication_month
10966,1200000,2018-09-10,06:28:40,55.012935,83.004523,9654,1,2,10,1,39.35,6.0,11,2018-09-10 06:28:40,2018,3,9
25141,8213280,2018-09-11,20:40:19,59.756070,30.365931,2661,2,3,4,3,96.40,13.0,11,2018-09-11 20:40:19,2018,3,9
42858,3960000,2018-09-14,08:36:58,55.961723,37.650433,81,2,6,6,3,69.80,9.4,11,2018-09-14 08:36:58,2018,3,9
42860,2900000,2018-09-14,08:36:58,55.961723,37.650433,81,2,1,6,2,50.30,11.2,11,2018-09-14 08:36:58,2018,3,9
45455,1718665,2018-09-14,14:19:50,55.009805,82.904909,9654,3,2,26,1,29.40,5.0,11,2018-09-14 14:19:50,2018,3,9
45457,2334437,2018-09-14,14:19:51,55.009805,82.904909,9654,3,26,26,1,38.47,5.0,11,2018-09-14 14:19:51,2018,3,9
45527,4690560,2018-09-14,14:32:13,55.014355,82.889340,9654,3,15,26,3,69.80,5.0,11,2018-09-14 14:32:13,2018,3,9
53967,2983100,2018-09-15,23:34:00,57.602566,39.857108,2604,3,7,7,2,64.85,11.0,11,2018-09-15 23:34:00,2018,3,9
55244,4400000,2018-09-16,07:45:21,54.965238,82.962853,9654,3,4,9,2,71.00,16.0,1,2018-09-16 07:45:21,2018,3,9
56202,2615000,2018-09-16,11:04:09,57.121232,65.487292,3991,2,1,10,2,53.00,9.0,1,2018-09-16 11:04:09,2018,3,9


In [81]:
df = df.drop_duplicates()

In [82]:
df.duplicated().sum()

np.int64(0)

## Validity checks

In [83]:
main_numeric_columns = [
    "price",
    "area",
    "kitchen_area",
    "rooms",
    "floor",
    "floors",
    "geo_lat",
    "geo_lon",
]

df[main_numeric_columns].describe()

,price,area,kitchen_area,rooms,floor,floors,geo_lat,geo_lon
count,5.475483e+06,5.475483e+06,5.475483e+06,5.475483e+06,5.475483e+06,5.475483e+06,5.475483e+06,5.475483e+06
mean,4.421478e+06,5.391823e+01,1.062817e+01,1.726213e+00,6.213816e+00,1.139749e+01,5.403784e+01,5.324654e+01
std,2.151016e+07,3.335522e+01,9.793369e+00,1.082136e+00,4.956792e+00,6.534970e+00,4.622977e+00,2.074788e+01
min,-2.144967e+09,7.000000e-02,1.000000e-02,-2.000000e+00,1.000000e+00,1.000000e+00,4.145906e+01,1.989020e+01
25%,1.950000e+06,3.800000e+01,7.000000e+00,1.000000e+00,2.000000e+00,5.000000e+00,5.337726e+01,3.777797e+01
50%,2.990000e+06,4.802000e+01,9.700000e+00,2.000000e+00,5.000000e+00,1.000000e+01,5.517128e+01,4.307021e+01
75%,4.800020e+06,6.313000e+01,1.270000e+01,2.000000e+00,9.000000e+00,1.600000e+01,5.622613e+01,6.564969e+01
max,2.147484e+09,7.856000e+03,9.999000e+03,1.000000e+01,3.900000e+01,3.900000e+01,7.198040e+01,1.625361e+02


In [84]:
invalid_counts = {
    "price <= 0": (df["price"] <= 0).sum(),
    "area <= 0": (df["area"] <= 0).sum(),
    "rooms < -1": (df["rooms"] < -1).sum(),
    "floor <= 0": (df["floor"] <= 0).sum(),
    "floors <= 0": (df["floors"] <= 0).sum(),
    "floor > floors": (df["floor"] > df["floors"]).sum(),
    "kitchen_area < 0": (df["kitchen_area"] < 0).sum(),
    "kitchen_area > area": (df["kitchen_area"] > df["area"]).sum(),
    "invalid geo_lat": (~df["geo_lat"].between(41, 82)).sum(),
    "invalid geo_lon": (~df["geo_lon"].between(19, 180)).sum(),
    "duplicates": df.duplicated().sum(),
}

invalid_counts = pd.Series(invalid_counts).sort_values(ascending=False)
invalid_counts

kitchen_area > area    3266
floor > floors         1072
price <= 0              388
rooms < -1              343
area <= 0                 0
floors <= 0               0
floor <= 0                0
kitchen_area < 0          0
invalid geo_lat           0
invalid geo_lon           0
duplicates                0
dtype: int64

## Cleaning

In [85]:
df_clean = df.copy()
rows_before_cleaning = df_clean.shape[0]

print("Rows before cleaning:", rows_before_cleaning)

Rows before cleaning: 5475483


In [86]:
df_clean = df_clean[
    (df_clean["price"] > 0)
    & (df_clean["area"] > 0)
    & (df_clean["rooms"] >= -1)
]

print("Rows after price, area, rooms cleaning:", df_clean.shape[0])

Rows after price, area, rooms cleaning: 5474752


In [87]:
df_clean = df_clean[
    (df_clean["floor"] > 0)
    & (df_clean["floors"] > 0)
    & (df_clean["floor"] <= df_clean["floors"])
]

print("Rows after floor cleaning:", df_clean.shape[0])

Rows after floor cleaning: 5473680


In [88]:
df_clean = df_clean[
    df_clean["kitchen_area"].isna()
    | (
        (df_clean["kitchen_area"] >= 0)
        & (df_clean["kitchen_area"] <= df_clean["area"])
    )
]

print("Rows after kitchen_area cleaning:", df_clean.shape[0])

Rows after kitchen_area cleaning: 5470541


In [89]:
df_clean = df_clean[
    df_clean["geo_lat"].between(41, 82)
    & df_clean["geo_lon"].between(19, 180)
]

print("Rows after geo cleaning:", df_clean.shape[0])

Rows after geo cleaning: 5470541


In [90]:
rows_after_cleaning = df_clean.shape[0]
removed_rows = rows_before_cleaning - rows_after_cleaning
removed_percent = round(removed_rows / rows_before_cleaning * 100, 2)

print("Rows before cleaning:", rows_before_cleaning)
print("Rows after cleaning:", rows_after_cleaning)
print("Removed rows:", removed_rows)
print("Removed percent:", removed_percent)

Rows before cleaning: 5475483
Rows after cleaning: 5470541
Removed rows: 4942
Removed percent: 0.09


In [91]:
print("price <= 0:", (df_clean["price"] <= 0).sum())
print("area <= 0:", (df_clean["area"] <= 0).sum())
print("rooms < -1:", (df_clean["rooms"] < -1).sum())
print("floor <= 0:", (df_clean["floor"] <= 0).sum())
print("floors <= 0:", (df_clean["floors"] <= 0).sum())
print("floor > floors:", (df_clean["floor"] > df_clean["floors"]).sum())
print("kitchen_area < 0:", (df_clean["kitchen_area"] < 0).sum())
print("kitchen_area > area:", (df_clean["kitchen_area"] > df_clean["area"]).sum())
print("invalid geo_lat:", (~df_clean["geo_lat"].between(41, 82)).sum())
print("invalid geo_lon:", (~df_clean["geo_lon"].between(19, 180)).sum())
print("duplicates:", df_clean.duplicated().sum())

price <= 0: 0
area <= 0: 0
rooms < -1: 0
floor <= 0: 0
floors <= 0: 0
floor > floors: 0
kitchen_area < 0: 0
kitchen_area > area: 0
invalid geo_lat: 0
invalid geo_lon: 0
duplicates: 0


In [92]:
df_clean.head()

,price,date,time,geo_lat,geo_lon,region,building_type,floor,floors,rooms,area,kitchen_area,object_type,published_at,publication_year,publication_quarter,publication_month
0,6050000,2018-02-19,20:00:21,59.805808,30.376141,2661,1,8,10,3,82.6,10.8,1,2018-02-19 20:00:21,2018,1,2
1,8650000,2018-02-27,12:04:54,55.683807,37.297405,81,3,5,24,2,69.1,12.0,1,2018-02-27 12:04:54,2018,1,2
2,4000000,2018-02-28,15:44:00,56.295250,44.061637,2871,1,5,9,3,66.0,10.0,1,2018-02-28 15:44:00,2018,1,2
3,1850000,2018-03-01,11:24:52,44.996132,39.074783,2843,4,12,16,2,38.0,5.0,11,2018-03-01 11:24:52,2018,1,3
4,5450000,2018-03-01,17:42:43,55.918767,37.984642,81,3,13,14,2,60.0,10.0,1,2018-03-01 17:42:43,2018,1,3


In [93]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 5470541 entries, 0 to 5477005
Data columns (total 17 columns):
 #   Column               Dtype         
---  ------               -----         
 0   price                int64         
 1   date                 str           
 2   time                 str           
 3   geo_lat              float64       
 4   geo_lon              float64       
 5   region               int64         
 6   building_type        int64         
 7   floor                int64         
 8   floors               int64         
 9   rooms                int64         
 10  area                 float64       
 11  kitchen_area         float64       
 12  object_type          int64         
 13  published_at         datetime64[us]
 14  publication_year     int32         
 15  publication_quarter  int32         
 16  publication_month    int32         
dtypes: datetime64[us](1), float64(4), int32(3), int64(7), str(2)
memory usage: 688.7 MB


## Save cleaned dataset

In [94]:
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Saved to: {PROCESSED_DATA_PATH}")
print("File exists:", PROCESSED_DATA_PATH.exists())

Saved to: c:\Users\mikri\OneDrive\Desktop\real-estate-price-advisor\data\processed\real_estate_clean.csv
File exists: True
